# Kronos ETF Fine-Tuning v2

Fine-tunes `NeoQuasar/Kronos-small` on free daily OHLCV data for **22 US ETFs** (broad index, sector SPDRs, international, bonds, commodities), pulled live from Yahoo Finance via `yfinance`. Holdouts (never trained): **VTI, XLF, VNQ**.

**v2 training recipe** vs v1: lower LR (3e-5) with cosine decay, step-level early stopping, EMA weights, fp16 mixed precision (AMP), and a wall-clock guard (~4.8 h) so the run finishes well inside a single GPU session and respects a 5 h GPU-time budget. Best checkpoint is chosen from whichever of raw / EMA weights has the lower validation loss.

No paid data or API keys required. Model code is cloned from the public [Kronos repo](https://github.com/shiyu-coder/Kronos) at runtime.

In [ ]:
!pip install -q yfinance pandas-market-calendars "datasets>=2.19" pyyaml einops safetensors huggingface_hub

In [ ]:
import subprocess
import sys
from pathlib import Path

KRONOS_DIR = Path("/kaggle/working/Kronos")
if not KRONOS_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/shiyu-coder/Kronos.git", str(KRONOS_DIR)],
        check=True,
    )
if str(KRONOS_DIR) not in sys.path:
    sys.path.insert(0, str(KRONOS_DIR))

from model import Kronos, KronosTokenizer  # noqa: E402

MODEL_NAME = "NeoQuasar/Kronos-small"
TOKENIZER_NAME = "NeoQuasar/Kronos-Tokenizer-base"
assert "amazon" not in MODEL_NAME.lower() and "chronos-t5" not in MODEL_NAME.lower(), \
    "Amazon Chronos model is not allowed in this project."
print("Kronos model code loaded OK")

## Step 1 - Build the multi-symbol ETF dataset (v2, 22 symbols)

Free daily OHLCV via Yahoo Finance, technical features, BULL/BEAR/SIDEWAYS/CRASH regime labels (ETF-scaled thresholds), then windowed into 380-bar (360 lookback + 20 prediction) samples with **step 4** (denser than v1's 5). Windows are bucketed into train/val/test by their **target** start date. Holdout symbols VTI/XLF/VNQ are excluded entirely.

In [ ]:
import math
import numpy as np
import pandas as pd

DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# v2 universe: 22 symbols. HOLDOUT (never trained): VTI, XLF, VNQ.
ETF_FINETUNE_UNIVERSE = [
    # v1 core
    "SPY", "QQQ", "IWM", "DIA", "EFA", "GLD", "TLT",
    # sector SPDRs (XLF intentionally excluded -> stays holdout)
    "XLK", "XLE", "XLV", "XLI", "XLP", "XLU", "XLY", "XLB",
    # international / bonds / commodities
    "EEM", "EWJ", "VGK", "IEF", "LQD", "HYG", "SLV",
]
HOLDOUT = {"VTI", "XLF", "VNQ"}
assert not (set(ETF_FINETUNE_UNIVERSE) & HOLDOUT), "Holdout leaked into training universe"

ETF_LOOKBACK = 360
ETF_PRED_LEN = 20
ETF_WINDOW_SIZE = ETF_LOOKBACK + ETF_PRED_LEN
ETF_WINDOW_STEP = 4
ETF_PERIOD = "15y"
ETF_CRASH_THRESHOLD = -0.07
ETF_BEAR_THRESHOLD = -0.015
TRAIN_CUTOFF = pd.Timestamp("2025-01-01", tz="UTC")
VAL_CUTOFF = pd.Timestamp("2025-10-01", tz="UTC")

FEATURE_COLUMNS = [
    "open", "high", "low", "close", "volume", "amount", "funding_rate",
    "rsi_14", "atr_14", "bb_width", "vol_zscore", "log_return",
    "realized_vol", "funding_8h_ma",
]
KRONOS_FEATURE_COLUMNS = ["open", "high", "low", "close", "volume", "amount"]


def status(message: str) -> None:
    print(f"[PREP] {message}", flush=True)


def fetch_etf_ohlcv(symbol: str, period: str = ETF_PERIOD, interval: str = "1d") -> pd.DataFrame:
    import yfinance as yf

    df = yf.download(symbol, period=period, interval=interval, auto_adjust=True, progress=False)
    if df is None or df.empty:
        raise RuntimeError(f"Yahoo Finance returned no data for {symbol}.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.rename(columns=str.lower)
    df = df.reset_index().rename(columns={"Date": "timestamp", "Datetime": "timestamp"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    numeric_cols = ["open", "high", "low", "close", "volume"]
    df[numeric_cols] = df[numeric_cols].astype(float)
    df["funding_rate"] = 0.0
    return df[["timestamp", *numeric_cols, "funding_rate"]].reset_index(drop=True)


def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["amount"] = out["volume"] * out[["open", "high", "low", "close"]].mean(axis=1)
    out["rsi_14"] = compute_rsi(out["close"], 14)
    prev_close = out["close"].shift(1)
    true_range = pd.concat(
        [out["high"] - out["low"], (out["high"] - prev_close).abs(), (out["low"] - prev_close).abs()],
        axis=1,
    ).max(axis=1)
    out["atr_14"] = true_range.rolling(14).mean()
    middle = out["close"].rolling(20).mean()
    std = out["close"].rolling(20).std()
    out["bb_width"] = ((middle + 2 * std) - (middle - 2 * std)) / middle
    vol_mean = out["volume"].rolling(168).mean()
    vol_std = out["volume"].rolling(168).std()
    out["vol_zscore"] = (out["volume"] - vol_mean) / vol_std.replace(0, np.nan)
    out["log_return"] = np.log(out["close"] / out["close"].shift(1))
    out["realized_vol"] = out["log_return"].rolling(24).std() * math.sqrt(24)
    out["funding_8h_ma"] = out["funding_rate"].rolling(3).mean()
    return out


def add_regimes(df: pd.DataFrame, crash_threshold: float, bear_threshold: float) -> pd.DataFrame:
    out = df.copy()
    out["price_change_7d"] = (out["close"] - out["close"].shift(168)) / out["close"].shift(168)
    conditions = [
        out["price_change_7d"] < crash_threshold,
        out["price_change_7d"] < bear_threshold,
        out["price_change_7d"] > -bear_threshold,
    ]
    out["regime"] = np.select(conditions, ["CRASH", "BEAR", "BULL"], default="SIDEWAYS")
    return out


def make_windows_by_target_split(df, window_size, lookback, step, train_cutoff, val_cutoff, symbol=""):
    train_records, val_records, test_records = [], [], []
    for start_idx in range(0, len(df) - window_size + 1, step):
        window = df.iloc[start_idx : start_idx + window_size]
        target_start = window["timestamp"].iloc[lookback]
        regime = str(window["regime"].iloc[-1])
        record = {
            "start": window["timestamp"].iloc[0].isoformat(),
            "features": window[KRONOS_FEATURE_COLUMNS].astype("float32").to_numpy().tolist(),
            "target": window["close"].astype("float32").to_numpy().tolist(),
            "regime": regime,
        }
        if symbol:
            record["symbol"] = symbol
        if target_start < train_cutoff:
            train_records.append(record)
        elif target_start < val_cutoff:
            val_records.append(record)
        else:
            test_records.append(record)
    return train_records, val_records, test_records


def balance_train_windows(records):
    if not records:
        return records, False
    balanced = list(records)
    applied = False
    target_min_pct = 0.15
    while True:
        counts = pd.Series([r["regime"] for r in balanced]).value_counts().to_dict()
        total = len(balanced)
        underrepresented = [
            r for r in ["CRASH", "BEAR", "BULL", "SIDEWAYS"] if counts.get(r, 0) / total < target_min_pct
        ]
        underrepresented = [r for r in underrepresented if any(rec["regime"] == r for rec in records)]
        if not underrepresented:
            break
        for regime in underrepresented:
            candidates = [r for r in records if r["regime"] == regime]
            balanced.extend(candidates)
            applied = True
        if len(balanced) > len(records) * 10:
            break
    return balanced, applied


all_train, all_val, all_test = [], [], []
for symbol in ETF_FINETUNE_UNIVERSE:
    status(f"Fetching {symbol} daily OHLCV from Yahoo Finance...")
    try:
        raw = fetch_etf_ohlcv(symbol)
    except Exception as exc:
        status(f"  [WARN] {symbol} skipped: {exc}")
        continue
    df = add_regimes(add_features(raw), ETF_CRASH_THRESHOLD, ETF_BEAR_THRESHOLD)
    df = df.dropna(subset=FEATURE_COLUMNS + ["regime"]).reset_index(drop=True)
    status(f"{symbol}: {len(df)} usable daily bars")
    train_records, val_records, test_records = make_windows_by_target_split(
        df, ETF_WINDOW_SIZE, ETF_LOOKBACK, ETF_WINDOW_STEP, TRAIN_CUTOFF, VAL_CUTOFF, symbol=symbol
    )
    all_train.extend(train_records)
    all_val.extend(val_records)
    all_test.extend(test_records)

balanced_train, oversampling_applied = balance_train_windows(all_train)

from datasets import Dataset

Dataset.from_list(balanced_train).save_to_disk(str(DATA_DIR / "etf_train.arrow"))
Dataset.from_list(all_val).save_to_disk(str(DATA_DIR / "etf_val.arrow"))
Dataset.from_list(all_test).save_to_disk(str(DATA_DIR / "etf_test.arrow"))

print("")
print(f"train : {len(balanced_train)} windows (oversampling_applied={oversampling_applied})")
print(f"val   : {len(all_val)} windows")
print(f"test  : {len(all_test)} windows")

## Step 2 - Fine-tune Kronos on the ETF windows (v2 recipe)

Lower LR + cosine decay, AMP fp16, EMA weights, step-level validation with early stopping, and a wall-clock guard. The best checkpoint (lower of raw / EMA val loss) is saved to `best_clean` incrementally, so an early stop or the wall-clock guard never loses progress.

In [ ]:
import math
import random
import shutil
import time
import json
from datetime import timedelta

import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset as TorchDataset
from datasets import load_from_disk

CONFIG = {
    "context_length": ETF_LOOKBACK,
    "prediction_length": ETF_PRED_LEN,
    "num_epochs": 6,                 # upper bound; early stopping / wall-clock decide
    "batch_size": 8,
    "gradient_accumulation_steps": 4,
    "learning_rate": 3.0e-5,         # v1 used 1e-4 and overfitted after epoch 1
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "seed": 42,
    "eval_every_steps": 150,         # optimizer steps between validation evals
    "early_stop_patience": 4,        # stop after N evals without improvement
    "ema_decay": 0.999,
    "max_hours": 4.8,                # wall-clock guard (keeps GPU time under ~5h)
}
OUTPUT_DIR = Path("/kaggle/working/checkpoints_etf")
BAR_STEP = timedelta(days=1)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Device: CPU (no GPU detected - check the notebook's accelerator setting)")


def make_time_features(start_iso: str, length: int, bar_step: timedelta) -> np.ndarray:
    start = pd.Timestamp(start_iso)
    timestamps = pd.Series([start + bar_step * i for i in range(length)])
    return np.column_stack(
        [
            timestamps.dt.minute.to_numpy(),
            timestamps.dt.hour.to_numpy(),
            timestamps.dt.weekday.to_numpy(),
            timestamps.dt.day.to_numpy(),
            timestamps.dt.month.to_numpy(),
        ]
    ).astype(np.float32)


class KronosWindowDataset(TorchDataset):
    def __init__(self, arrow_data, context_length, prediction_length, clip=5.0, bar_step=BAR_STEP):
        self.data = arrow_data
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.window_length = context_length + prediction_length
        self.clip = clip
        self.bar_step = bar_step

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        record = self.data[int(idx)]
        features = np.asarray(record["features"], dtype=np.float32)[: self.window_length]
        if features.shape[0] < self.window_length:
            raise RuntimeError(f"Expected at least {self.window_length} rows, got {features.shape[0]}")
        context = features[: self.context_length]
        mean = context.mean(axis=0)
        std = context.std(axis=0)
        normalized = np.clip((features - mean) / (std + 1e-5), -self.clip, self.clip).astype(np.float32)
        stamps = make_time_features(str(record["start"]), len(normalized), self.bar_step)
        return {"x": torch.from_numpy(normalized), "stamp": torch.from_numpy(stamps)}


def collate(batch):
    return {
        "x": torch.stack([item["x"] for item in batch]),
        "stamp": torch.stack([item["stamp"] for item in batch]),
    }


train_data = load_from_disk(str(DATA_DIR / "etf_train.arrow"))
val_data = load_from_disk(str(DATA_DIR / "etf_val.arrow"))

train_ds = KronosWindowDataset(train_data, CONFIG["context_length"], CONFIG["prediction_length"])
val_ds = KronosWindowDataset(val_data, CONFIG["context_length"], CONFIG["prediction_length"])
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate)

print(f"train batches/epoch: {len(train_loader)} | val batches: {len(val_loader)}")

tokenizer = KronosTokenizer.from_pretrained(TOKENIZER_NAME).to(device).eval()
model = Kronos.from_pretrained(MODEL_NAME).to(device)
model.train()

use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def kronos_loss(batch):
    x = batch["x"].to(device, non_blocking=True)
    stamp = batch["stamp"].to(device, non_blocking=True)
    with torch.no_grad():
        token_s1, token_s2 = tokenizer.encode(x, half=True)
    token_in = [token_s1[:, :-1], token_s2[:, :-1]]
    token_out = [token_s1[:, 1:], token_s2[:, 1:]]
    with torch.cuda.amp.autocast(enabled=use_amp):
        logits = model(token_in[0], token_in[1], stamp[:, :-1, :])
        loss, _, _ = model.head.compute_loss(logits[0], logits[1], token_out[0], token_out[1])
    return loss


def evaluate_current():
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in val_loader:
            losses.append(float(kronos_loss(batch).detach().cpu()))
    model.train()
    return float(np.mean(losses)) if losses else math.inf


# --- EMA (shadow copy of floating-point params/buffers) ---
ema_state = {k: v.detach().clone() for k, v in model.state_dict().items()}


def update_ema(decay):
    with torch.no_grad():
        msd = model.state_dict()
        for k, v in msd.items():
            if v.dtype.is_floating_point:
                ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)
            else:
                ema_state[k].copy_(v)


def evaluate_state(state_dict):
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state_dict)
    loss = evaluate_current()
    model.load_state_dict(backup)
    return loss


def save_state_as_checkpoint(state_dict, out_dir):
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state_dict)
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    model.load_state_dict(backup)


optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
accumulation = CONFIG["gradient_accumulation_steps"]
steps_per_epoch = max(1, math.ceil(len(train_loader) / accumulation))
total_steps = max(1, steps_per_epoch * CONFIG["num_epochs"])
warmup_steps = CONFIG["warmup_steps"]


def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-8, step / max(1, warmup_steps))
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(1.0, max(0.0, progress))
    return 0.5 * (1.0 + math.cos(math.pi * progress))  # cosine decay to 0


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_dir = OUTPUT_DIR / "best_clean"
best_val = math.inf
best_is_ema = False
best_step = 0
evals_without_improve = 0
global_step = 0
stop_training = False
start_time = time.time()


def maybe_evaluate_and_checkpoint():
    global best_val, best_is_ema, best_step, evals_without_improve, stop_training
    raw_val = evaluate_current()
    ema_val = evaluate_state(ema_state)
    cur_is_ema = ema_val < raw_val
    cur_best = min(raw_val, ema_val)
    elapsed_h = (time.time() - start_time) / 3600.0
    improved = cur_best < best_val - 1e-6
    if improved:
        best_val = cur_best
        best_is_ema = cur_is_ema
        best_step = global_step
        evals_without_improve = 0
        save_state_as_checkpoint(ema_state if cur_is_ema else model.state_dict(), best_dir)
    else:
        evals_without_improve += 1
    print(
        f"[EVAL] step={global_step} raw_val={raw_val:.4f} ema_val={ema_val:.4f} "
        f"best={best_val:.4f}({'ema' if best_is_ema else 'raw'}) "
        f"no_improve={evals_without_improve} elapsed={elapsed_h:.2f}h",
        flush=True,
    )
    if evals_without_improve >= CONFIG["early_stop_patience"]:
        print("[STOP] early-stopping patience exhausted.", flush=True)
        stop_training = True
    if elapsed_h >= CONFIG["max_hours"]:
        print(f"[STOP] wall-clock guard: {elapsed_h:.2f}h >= {CONFIG['max_hours']}h.", flush=True)
        stop_training = True


for epoch in range(CONFIG["num_epochs"]):
    if stop_training:
        break
    epoch_losses = []
    optimizer.zero_grad(set_to_none=True)
    for batch_idx, batch in enumerate(train_loader, start=1):
        loss = kronos_loss(batch) / accumulation
        scaler.scale(loss).backward()
        epoch_losses.append(float(loss.detach().cpu()) * accumulation)

        if batch_idx % accumulation == 0 or batch_idx == len(train_loader):
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            update_ema(CONFIG["ema_decay"])
            global_step += 1

            if global_step % CONFIG["eval_every_steps"] == 0:
                maybe_evaluate_and_checkpoint()
                if stop_training:
                    break

        if batch_idx % 50 == 0:
            lr = scheduler.get_last_lr()[0]
            print(
                f"[TRAIN] epoch={epoch + 1}/{CONFIG['num_epochs']} batch={batch_idx}/{len(train_loader)} "
                f"loss={epoch_losses[-1]:.4f} lr={lr:.2e}",
                flush=True,
            )

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else math.inf
    print(f"Epoch {epoch + 1} done | train_loss={train_loss:.4f}", flush=True)

# Final eval in case the run ended between eval points (and never checkpointed).
if best_val == math.inf:
    maybe_evaluate_and_checkpoint()

meta = {
    "best_val_loss": best_val,
    "best_step": best_step,
    "best_is_ema": best_is_ema,
    "total_steps_run": global_step,
    "wall_clock_hours": (time.time() - start_time) / 3600.0,
    "config": CONFIG,
    "universe": ETF_FINETUNE_UNIVERSE,
    "window_step": ETF_WINDOW_STEP,
    "train_windows": len(train_ds),
    "val_windows": len(val_ds),
}
(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(OUTPUT_DIR / "training_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)

print("")
print(f"Training complete. best_val_loss={best_val:.4f} @ step {best_step} "
      f"(ema={best_is_ema}). Saved: {best_dir}")
print(json.dumps(meta, indent=2))

## Output

The fine-tuned checkpoint is at `/kaggle/working/checkpoints_etf/best_clean` with `training_meta.json` alongside. Download the kernel output, keep the old checkpoint as `finetune/checkpoints_etf/best_clean_v1`, then drop the new one into `finetune/checkpoints_etf/best_clean` and re-run `calibrate.py` + backtests.

In [ ]:
import subprocess
print(subprocess.run(["find", "/kaggle/working/checkpoints_etf", "-maxdepth", "2"], capture_output=True, text=True).stdout)
try:
    print(open("/kaggle/working/checkpoints_etf/training_meta.json").read())
except FileNotFoundError:
    print("training_meta.json not found")